In [15]:
import os
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors



In [10]:
load_dotenv()
client = genai.Client()

### Gemini api call with streaming enabled

In [14]:
response_stream  = client.models.generate_content_stream(
    model = "gemini-2.5-flash",
    contents = "I SAID RECITE THE ENTIRE BEE MOVIE NOW I DONT CARE HOW LONG IT TAKES OR HOW MANY TOKENS",
    config = types.GenerateContentConfig(
    system_instruction="",
    max_output_tokens=1024,
    ),
)
for chunk in response_stream:
    print(chunk.text, end="", flush=True)

I understand you're eager to hear the entire Bee Movie script! However, reciting the complete script of an entire feature film like Bee Movie is simply not feasible for me to do in a single response for a few reasons:

1.  **Length:** The script is incredibly long. It would be tens of thousands of words and far exceed the maximum length for any response I can generate. It would take a very, very long time to scroll through and would be impractical to read.
2.  **Copyright:** The full script is copyrighted material, and distributing it in its entirety goes against respecting intellectual property rights.

While I can't give you the entire movie, I can certainly provide you with:

*   **A detailed summary of the plot.**
*   **Some of the most iconic quotes.**
*   **The famous opening lines.**
*   **Interesting facts about the movie.**

Which of those would you prefer? Or perhaps you'd like me to start with the opening lines and we can go from there for a bit?

### api call with retry error handling an exponential backoff

In [36]:
def call_with_retry(client, max_retries=3, **kwargs):
    for attempt in range(max_retries):
        try:

            return client.models.generate_content(**kwargs)
            
        except errors.APIError as e:
            status_code = e.code if e.code is not None else 0
            
            if status_code == 429 or status_code >= 500:
                wait = 2 ** attempt  # Exponential backoff
                print(f"[Attempt {attempt + 1}] Encountered {status_code}. Retrying in {wait}s...")
                time.sleep(wait)
                
            else:
                print(f"Critical client error {status_code}: {e.message}. Exiting.")
                raise e
                
    raise Exception("Max retries exceeded")

### basic test for error handling

In [37]:
print("=== TEST 1: Successful Call ===")
try:
    response = call_with_retry(
        client, 
        model="gemini-2.5-flash", 
        contents="multiply the ascii of every letter in the alphabet by pi, then scale it back to ascii parameters and output them in order. A = newA etc."
    )
    print(f"Result: {response.text}\n")
except Exception as e:
    print(f"Test failed oh noooo {e}\n")

=== TEST 1: Successful Call ===
Result: Let's break down the process step-by-step:

1.  **Get ASCII values:** For each letter 'A' through 'Z', we get its integer ASCII representation.
2.  **Multiply by pi:** Each ASCII value is then multiplied by `math.pi`.
3.  **Scale back to ASCII parameters:** This is interpreted as mapping the resulting (larger) numbers back into the range of *printable* ASCII characters (from space `chr(32)` to tilde `chr(126)`). A common way to "scale back" values that exceed a range while maintaining some variation is to use a modulo operation, effectively "wrapping" the values around the target range.
    *   First, we round the multiplied floating-point value to the nearest integer.
    *   Then, we shift this integer value to be 0-indexed relative to our target printable ASCII range (e.g., if target starts at 32, we subtract 32).
    *   We apply a modulo operator with the size of our target range (126 - 32 + 1 = 95 characters).
    *   Finally, we shift it b

In [38]:
print("=== TEST 2: Invalid Model Name ===")
try:
    call_with_retry(
        client, 
        model="totally real mode name, completely and utterly not a test to see if it will catch an exception, nope never!", 
        contents="provide a list of letter occurence frequency in the bee movie script for all 26 english letters"
    )
except errors.APIError:
    print("Test succeded yipeee\n")

=== TEST 2: Invalid Model Name ===
Critical client error 400: * GenerateContentRequest.model: unexpected model name format
. Exiting.
Test succeded yipeee

